In [1]:
healthy = ["/dsi/sbm/chen2/raw_data/healthy/TRB_h_v1", "/dsi/sbm/chen2/raw_data/healthy/TRB_h_v2"]
sick = "/dsi/sbm/chen2/raw_data/sick/mixcr_4.7_TRB"

downsample_path = "/dsi/sbm/chen2/downsampled_data"

In [2]:
def prepare_data(data, cdr3_header):

    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    return data_filtered

In [6]:
import pandas as pd
from glob import glob
from collections import Counter
import chephy_model as cpm
from tqdm import tqdm

# ---------- CONFIG ----------
cdr3_header = 'aaSeqCDR3'  # Adjust if your column is different
downsample_size = 10000

# ---------- 1. Load patient files ----------
def load_patient_files(paths, cdr3_header='CDR3'):
    patient_dfs = []
    for path in paths if isinstance(paths, list) else [paths]:
        files = glob(f"{path}/*.tsv")
        for f in tqdm(files):
            df = pd.read_csv(f, sep='\t', low_memory=False)
            df['patient_id'] = f  # tag patient by file path
            patient_dfs.append(df)
    return patient_dfs

# Replace with your folder paths
healthy_patients = load_patient_files(healthy, cdr3_header)
sick_patients = load_patient_files(sick, cdr3_header)

# ---------- 2. Prepare + Truncate BEFORE public sequence detection ----------
def prepare_and_truncate(df, cdr3_header):
    df = prepare_data(df, cdr3_header)
    df = cpm.truncate_sequences(df, cdr3_header, 4, 4)
    return df

healthy_prepared = [prepare_and_truncate(df, cdr3_header) for df in healthy_patients]
sick_prepared = [prepare_and_truncate(df, cdr3_header) for df in sick_patients]
all_prepared = healthy_prepared + sick_prepared

cdr3_header_truncated = "cdr3_truncated"  # Update header after truncation
# ---------- 3. Identify public sequences (after truncation) ----------
seq_patient_map = {}
for df in all_prepared:
    pid = df['patient_id'].iloc[0]
    for seq in df[cdr3_header_truncated].dropna().unique():
        seq_patient_map.setdefault(seq, set()).add(pid)

public_seqs = {seq for seq, pids in seq_patient_map.items() if len(pids) > 1}

0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]


In [4]:
# ---------- 4. Final patient processing ----------
def process_patient(df, cdr3_header, public_seqs, sample=True):
    df = prepare_data(df, cdr3_header)
    df = cpm.truncate_sequences(df, cdr3_header, 4, 4)
    df = df[~df['cdr3_truncated'].isin(public_seqs)]
    df = df.drop_duplicates(subset='cdr3_truncated')
    
    if not sample:
        return df if len(df) > 0 else None
    
    if len(df) >= downsample_size:
        return df.sample(downsample_size, random_state=42, weights='readFraction')
    else:
        return None

full = False

# ---------- 5. Process healthy/sick datasets ----------
healthy_processed = [process_patient(df, cdr3_header, public_seqs, sample=full) for df in healthy_patients]
sick_processed = [process_patient(df, cdr3_header, public_seqs, sample=full) for df in sick_patients]


# ---------- 6. Filter out patients with <8000 usable sequences ----------
healthy_processed = [df for df in healthy_processed if df is not None]
sick_processed = [df for df in sick_processed if df is not None]

In [5]:
n = 1000
healthy_combined = pd.concat(healthy_processed, ignore_index=True)
sick_combined = pd.concat(sick_processed, ignore_index=True)
full_dataset = pd.concat([healthy_combined, sick_combined], ignore_index=True)
inverse_weights = False


ValueError: No objects to concatenate

In [ ]:
import numpy as np

# To take sequences with lower probability, sample using the inverse of 'readFraction' as weights.
# Add a small epsilon to avoid division by zero.

if inverse_weights:
    epsilon = 1e-10
    inv_weights = 1 / (sick_combined['readFraction'] + epsilon)
    inv_weights = inv_weights / inv_weights.sum()  # Normalize

    sick_sample = sick_combined.sample(
        n=n,
        weights=inv_weights,
        random_state=42
    )

    epsilon = 1e-10
    inv_weights = 1 / (healthy_combined['readFraction'] + epsilon)
    inv_weights = inv_weights / inv_weights.sum()  # Normalize

    # Sample 20k sequences from healthy patients
    healthy_sample = healthy_combined.sample(
        n=n,
        weights=inv_weights,
        random_state=42
    )


else:
   # Sample 20k sequences from sick patients
    sick_sample = sick_combined.sample(
        n=n, 
        weights='readFraction', 
        random_state=42
    )

    # Sample 20k sequences from healthy patients
    healthy_sample = healthy_combined.sample(
        n=n, 
        weights='readFraction', 
        random_state=42
    ) 



full_downsampled = pd.concat([sick_sample, healthy_sample], ignore_index=True)

In [ ]:
import gnn_utils
from scipy.spatial.distance import squareform

sequences = set(full_downsampled["cdr3_truncated"])

chephy_matrix, sequences_list = cpm.find_che_phy_dist(sequences)

full_downsampled['tcr_index'] = full_downsampled['cdr3_truncated'].apply(lambda x: sequences_list.index(x) if x in sequences_list else -1)
node_weights = full_downsampled.groupby(full_downsampled['tcr_index'])["readFraction"].sum().tolist()

full_downsampled["label"] = full_downsampled["patient_id"].apply(lambda x: 1 if str(sick) in str(x) else 0)
labels = full_downsampled["label"].tolist()

distance_threshold = 0.05
use_mst = True
enrich = True

In [ ]:
graph = gnn_utils.build_whole_graph(
                        squareform(chephy_matrix),
                        sequences_list,
                        labels=labels,
                        node_weights=node_weights,
                        distance_threshold=distance_threshold,
                        use_mst=use_mst,
                        enrich=enrich  
                    )

In [ ]:
# Export edge data
edge_list = []
for source_idx, target_idx in graph.edge_index.T:
    source_seq = sequences_list[source_idx]
    target_seq = sequences_list[target_idx]
    edge_list.append({'source': source_seq, 'target': target_seq})

edge_df = pd.DataFrame(edge_list)
edge_df.to_csv("edge_data.csv", index=False)

# Export node data
node_list = []
for idx, seq in enumerate(sequences_list):
    # Find all rows in full_downsampled with this sequence
    label_rows = full_downsampled[full_downsampled['cdr3_truncated'] == seq]
    # If any label==1, mark as sick (red), else healthy (green)
    color = "red" if (label_rows["label"] == 1).any() else "green"
    node_list.append({'id': seq, 'color': color})

node_df = pd.DataFrame(node_list)
node_df.to_csv("node_data.csv", index=False)

In [ ]:
import random

sample_size = 250


# Find sick and healthy sequences not in full_downsampled
downsampled_set = set(full_downsampled['cdr3_truncated'])

# Sick: concatenate all sick_combined sequences, remove those in downsampled_set
sick_all = set(sick_combined['cdr3_truncated'])
sick_not_in_downsampled = list(sick_all - downsampled_set)

# Healthy: concatenate all healthy_combined sequences, remove those in downsampled_set
healthy_all = set(healthy_combined['cdr3_truncated'])
healthy_not_in_downsampled = list(healthy_all - downsampled_set)

# Sample 500 from each (if possible)
random.seed(42)
sick_extra = random.sample(sick_not_in_downsampled, min(sample_size, len(sick_not_in_downsampled)))
healthy_extra = random.sample(healthy_not_in_downsampled, min(sample_size, len(healthy_not_in_downsampled)))

# Labels: 1 for sick, 0 for healthy
extra_sequences = sick_extra + healthy_extra
extra_labels = [1]*len(sick_extra) + [0]*len(healthy_extra)

In [ ]:
import torch.nn as nn
from torch_geometric.nn import SAGEConv

class GraphSAGEClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, num_classes)
        self.relu = nn.LeakyReLU()
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x


In [ ]:
import torch

from sklearn.model_selection import train_test_split

all_indices = torch.arange(graph.num_nodes)
train_idx, val_idx = train_test_split(all_indices, test_size=0.2, stratify=graph.y)

# Masks for loss computation
train_mask = torch.zeros(graph.num_nodes, dtype=torch.bool)
val_mask = torch.zeros(graph.num_nodes, dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
graph.train_mask = train_mask
graph.val_mask = val_mask

In [ ]:
best_val_acc = 0.0
best_model_state = None

for epoch in range(1000):
    model.train()
    optimizer.zero_grad()
    out = model(graph.x, graph.edge_index)

    loss = loss_fn(out[train_mask], graph.y[train_mask])
    loss.backward()
    optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        val_out = out[val_mask]
        val_labels = graph.y[val_mask]
        pred = val_out.argmax(dim=1)
        correct = (pred == val_labels).sum().item()
        acc = correct / val_mask.sum().item()

    if acc > best_val_acc:
        best_val_acc = acc
        best_model_state = model.state_dict()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Val Acc: {acc:.4f}")

# Save the best model after training
if best_model_state is not None:
    torch.save(best_model_state, "best_graphsage_model.pt")
    print(f"Best model saved with val acc: {best_val_acc:.4f}")

Epoch 0, Loss: 0.6680, Val Acc: 0.5000
Epoch 10, Loss: 0.6649, Val Acc: 0.4775
Epoch 20, Loss: 0.6592, Val Acc: 0.5000
Epoch 30, Loss: 0.6543, Val Acc: 0.4925
Epoch 40, Loss: 0.6527, Val Acc: 0.5150
Epoch 50, Loss: 0.6485, Val Acc: 0.4825
Epoch 60, Loss: 0.6535, Val Acc: 0.5075
Epoch 70, Loss: 0.6290, Val Acc: 0.4975
Epoch 80, Loss: 0.6403, Val Acc: 0.4925
Epoch 90, Loss: 0.6373, Val Acc: 0.4900
Epoch 100, Loss: 0.6292, Val Acc: 0.5050
Epoch 110, Loss: 0.6235, Val Acc: 0.5025
Epoch 120, Loss: 0.6247, Val Acc: 0.5225
Epoch 130, Loss: 0.6216, Val Acc: 0.4975
Epoch 140, Loss: 0.6293, Val Acc: 0.4975
Epoch 150, Loss: 0.6086, Val Acc: 0.5075
Epoch 160, Loss: 0.6147, Val Acc: 0.5000
Epoch 170, Loss: 0.6068, Val Acc: 0.5050
Epoch 180, Loss: 0.6063, Val Acc: 0.5075
Epoch 190, Loss: 0.5990, Val Acc: 0.5175
Epoch 200, Loss: 0.6138, Val Acc: 0.5075
Epoch 210, Loss: 0.6426, Val Acc: 0.5200
Epoch 220, Loss: 0.6110, Val Acc: 0.4725
Epoch 230, Loss: 0.5955, Val Acc: 0.4975
Epoch 240, Loss: 0.5950, Va

In [ ]:
# Evaluation using extra_sequences and extra_labels
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

# Set model to evaluation mode
best_model_state.eval()

# Get model predictions for training nodes (to establish baseline)
with torch.no_grad():
    train_logits = model(graph.x, graph.edge_index)
    train_probs = torch.softmax(train_logits, dim=1)
    train_predictions = torch.argmax(train_probs, dim=1)

print(f"Training accuracy: {accuracy_score(graph.y.cpu(), train_predictions.cpu()):.4f}")

AttributeError: 'collections.OrderedDict' object has no attribute 'eval'

In [ ]:
# Evaluation for extra sequences
print(f"Evaluating on {len(extra_sequences)} extra sequences...")

# Compute distances between extra sequences and trained graph sequences
def predict_extra_sequences(extra_seqs, extra_labels, k=5):
    """
    Predict labels for extra sequences based on k-nearest neighbors in the trained graph
    """
    predictions = []
    
    for i, extra_seq in enumerate(tqdm(extra_seqs, desc="Processing extra sequences")):
        # Create a combined set of sequences: this extra sequence + all graph sequences
        combined_sequences = [extra_seq] + sequences_list
        
        # Compute distance matrix for combined sequences
        distance_matrix, _ = cpm.find_che_phy_dist(set(combined_sequences))
        distance_matrix = squareform(distance_matrix)
        
        # Get distances from extra sequence (index 0) to all graph sequences (indices 1:)
        distances = distance_matrix[0, 1:]
        
        # Find k nearest neighbors
        k_nearest_indices = np.argsort(distances)[:k]
        
        # Get predictions from k nearest neighbors using the trained model
        neighbor_predictions = train_predictions[k_nearest_indices].cpu().numpy()
        neighbor_probs = train_probs[k_nearest_indices].cpu().numpy()
        
        # Weight by inverse distance (closer neighbors have more influence)
        weights = 1.0 / (distances[k_nearest_indices] + 1e-8)
        weights = weights / weights.sum()
        
        # Weighted voting
        weighted_prob_sick = np.sum(weights * neighbor_probs[:, 1])
        prediction = 1 if weighted_prob_sick > 0.5 else 0
        predictions.append(prediction)
    
    return np.array(predictions)

# Make predictions for extra sequences
extra_predictions = predict_extra_sequences(extra_sequences, extra_labels, k=5)

Evaluating on 500 extra sequences...


Processing extra sequences:   0%|          | 0/500 [00:00<?, ?it/s]

Processing extra sequences: 100%|██████████| 500/500 [01:14<00:00,  6.72it/s]


In [ ]:
# Calculate evaluation metrics
extra_labels_array = np.array(extra_labels)
accuracy = accuracy_score(extra_labels_array, extra_predictions)
print(f"\n=== EVALUATION RESULTS ===")
print(f"Test accuracy on extra sequences: {accuracy:.4f}")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(extra_labels_array, extra_predictions, 
                          target_names=['Healthy', 'Sick']))

# Confusion Matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(extra_labels_array, extra_predictions)
print(cm)

# Additional metrics by class
from sklearn.metrics import precision_recall_fscore_support
precision, recall, f1, support = precision_recall_fscore_support(
    extra_labels_array, extra_predictions, average=None
)

print(f"\nDetailed Metrics:")
print(f"Healthy - Precision: {precision[0]:.4f}, Recall: {recall[0]:.4f}, F1: {f1[0]:.4f}")
print(f"Sick    - Precision: {precision[1]:.4f}, Recall: {recall[1]:.4f}, F1: {f1[1]:.4f}")

# Distribution of predictions
unique, counts = np.unique(extra_predictions, return_counts=True)
print(f"\nPrediction distribution:")
for val, count in zip(unique, counts):
    label_name = "Healthy" if val == 0 else "Sick"
    print(f"{label_name}: {count} ({count/len(extra_predictions)*100:.1f}%)")

print(f"\nTrue label distribution:")
unique, counts = np.unique(extra_labels_array, return_counts=True)
for val, count in zip(unique, counts):
    label_name = "Healthy" if val == 0 else "Sick"
    print(f"{label_name}: {count} ({count/len(extra_labels_array)*100:.1f}%)")


=== EVALUATION RESULTS ===
Test accuracy on extra sequences: 0.5020

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.67      0.01      0.02       250
        Sick       0.50      1.00      0.67       250

    accuracy                           0.50       500
   macro avg       0.58      0.50      0.34       500
weighted avg       0.58      0.50      0.34       500


Confusion Matrix:
[[  2 248]
 [  1 249]]

Detailed Metrics:
Healthy - Precision: 0.6667, Recall: 0.0080, F1: 0.0158
Sick    - Precision: 0.5010, Recall: 0.9960, F1: 0.6667

Prediction distribution:
Healthy: 3 (0.6%)
Sick: 497 (99.4%)

True label distribution:
Healthy: 250 (50.0%)
Sick: 250 (50.0%)
